<h2 align="center">Machine Learning Model Development</h2>

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    BaggingRegressor,
    VotingRegressor
)
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from sklearn.model_selection import GridSearchCV
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [7]:
df = pd.read_excel("../../data/processed/second_one_year_mine_data_cleaned.xlsx")
df.head()

,Date,PM10,PM2.5,Temperature,Relative_humidity,RF-CUMULATIVE,Wind_speed,Road_width,Distance between the dust generation/source point and the haul road monitoring station,Production_ROM,Production_OB,Production_ROM+OB,Vehicle capacity,LOCATION,SEASON
0,2025-03-04,70.8,24.1,36.1,20.4,0.0,8.0,7,20,30000,75000,105000,2.5,Near_Office,Summer
1,2025-03-08,69.5,32.6,36.0,22.4,0.0,8.0,7,20,30000,75000,105000,2.5,Near_Office,Summer
2,2025-03-11,65.2,30.4,35.5,21.8,0.0,8.0,7,20,30000,75000,105000,2.5,Near_Office,Summer
3,2025-03-15,76.8,26.5,34.8,38.5,0.0,6.0,7,20,30000,75000,105000,2.5,Near_Office,Summer
4,2025-03-18,58.4,20.4,35.1,31.4,0.0,7.0,7,20,30000,75000,105000,2.5,Near_Office,Summer


In [8]:
features = [
        "Temperature",
        "Relative_humidity",
        "RF-CUMULATIVE",
        "Wind_speed",
        "Road_width",
        "Distance between the dust generation/source point and the haul road monitoring station",
        "Production_ROM",
        "Production_OB",
        "Production_ROM+OB",
        "Vehicle capacity",
        "LOCATION",
        "SEASON"
]

target_pm10 = "PM10"
target_pm25 = "PM2.5"

In [11]:
X = df[features]
y = df[target_pm10]

X = pd.get_dummies(X, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

<h2 align="center">Linear Regression Model</h2>

In [12]:
lr = LinearRegression()
lr.fit(X_train, y_train)
pred = lr.predict(X_test)

In [13]:
mae = mean_absolute_error(y_test, pred)
rmse = np.sqrt(mean_squared_error(y_test, pred))
r2 = r2_score(y_test, pred)

print(mae)
print(rmse)
print(r2)

6.838789935827868
8.443220549095997
0.2232635431898774


<h2 align="center">Decision Tree Model</h2>

In [15]:
# Decision Tree
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

In [16]:
mae = mean_absolute_error(y_test, dt_pred)
rmse = np.sqrt(mean_squared_error(y_test, dt_pred))
r2 = r2_score(y_test, dt_pred)

print(mae)
print(rmse)
print(r2)

9.477586206896552
11.943220555670036
-0.5541764609519446


<h2 align="center">Random Forest Model</h2>

In [17]:
# Random Forest
rf = RandomForestRegressor(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)

In [18]:
mae = mean_absolute_error(y_test, rf_pred)
rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
r2 = r2_score(y_test, rf_pred)

print(mae)
print(rmse)
print(r2)

7.962728448275857
9.922011559011365
-0.07264714035885222


<h2 align="center">Bagging Model</h2>

In [19]:
# Bagging
try:
    bagging = BaggingRegressor(
        estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42
    )
except TypeError:
    bagging = BaggingRegressor(
        base_estimator=DecisionTreeRegressor(random_state=42),
        n_estimators=100,
        random_state=42
    )

bagging.fit(X_train, y_train)
bagging_pred = bagging.predict(X_test)

In [20]:
mae = mean_absolute_error(y_test, bagging_pred)
rmse = np.sqrt(mean_squared_error(y_test, bagging_pred))
r2 = r2_score(y_test, bagging_pred)

print(mae)
print(rmse)
print(r2)

8.079931034482762
10.102002013940373
-0.11191688051682402


<h2 align="center">SVR Model</h2>

In [21]:
# SVR
svr = SVR(kernel="rbf")
svr.fit(X_train, y_train)
svr_pred = svr.predict(X_test)

In [22]:
mae = mean_absolute_error(y_test, svr_pred)
rmse = np.sqrt(mean_squared_error(y_test, svr_pred))
r2 = r2_score(y_test, svr_pred)

print(mae)
print(rmse)
print(r2)

7.6019043496484855
9.459486984149194
0.025026997722050415


<h2 align="center">XGBoost Model</h2>

In [23]:
# XGBoost
xgb_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=200,
    learning_rate=0.1,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

In [24]:
mae = mean_absolute_error(y_test, xgb_pred)
rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
r2 = r2_score(y_test, xgb_pred)

print(mae)
print(rmse)
print(r2)

7.6457871338416785
9.59605894579972
-0.0033287017602574576


In [25]:
# Compare model performance in a table

results = []

models = {
    "Linear Regression": pred,
    "Decision Tree": dt_pred,
    "Random Forest": rf_pred,
    "Bagging": bagging_pred,
    "SVR": svr_pred,
    "XGBoost": xgb_pred,
}

for model_name, y_pred in models.items():
    results.append({
        "Model": model_name,
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "R²": r2_score(y_test, y_pred)
    })

# Voting Regressor
try:
    voting = VotingRegressor([
        ("lr", LinearRegression()),
        ("dt", DecisionTreeRegressor(random_state=42)),
        ("rf", RandomForestRegressor(n_estimators=200, random_state=42)),
        ("xgb", xgb.XGBRegressor(
            objective="reg:squarederror",
            n_estimators=200,
            learning_rate=0.1,
            max_depth=4,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42
        ))
    ])
    voting.fit(X_train, y_train)
    voting_pred = voting.predict(X_test)

    results.append({
        "Model": "Voting",
        "MAE": mean_absolute_error(y_test, voting_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, voting_pred)),
        "R²": r2_score(y_test, voting_pred)
    })
except Exception as e:
    print("Voting model could not be evaluated:", e)

results_df = pd.DataFrame(results)
results_df

,Model,MAE,RMSE,R²
0,Linear Regression,6.838790,8.443221,0.223264
1,Decision Tree,9.477586,11.943221,-0.554176
2,Random Forest,7.962728,9.922012,-0.072647
3,Bagging,8.079931,10.102002,-0.111917
4,SVR,7.601904,9.459487,0.025027
5,XGBoost,7.645787,9.596059,-0.003329
6,Voting,7.314106,9.269881,0.063720


In [26]:
Path("../../data/processed").mkdir(parents=True, exist_ok=True)
results_df.to_excel(
    "../../data/processed/secondoneyeardata_results_data2.xlsx",
    index=False
)